<div dir="rtl" lang="he" style="text-align: right;">

# מפגש 22 — Pipeline ו־Cross Validation
2 שעות הוראה ותרגול ו־1 שעות פרויקט; יחד 135 דקות.

## סיפור המסגרת
בהשוואת שיטות לא נרצה שמועמד אחד ייבחן בשאלות קלות ואחר בקשות. נשווה באותם קיפולים ונלמד הכנה מחדש בכל קיפול.

## מטרות
- להסביר קיפולי CV.
- למקם הכנה ב־Pipeline.
- לבחור ולהתאים מחדש.

</div>

<div dir="rtl" lang="he" style="text-align: right;">

## המושגים הנחוצים
CV מחלק את Train לתת־אימון ולאימות במספר מחזורים. Pipeline שומרת את ההכנה בתוך התהליך, כך שכל fit נעשה בתת־האימון המתאים. ממוצע CV משמש להשוואה ואינו Test חיצוני. לאחר הבחירה מתאימים מחדש את המתכון לכל Train.

</div>

<div dir="rtl" lang="he" style="text-align: right;">

## דוגמה פתורה לפני קוד
הציונים 0.8,0.9,0.7 נותנים ממוצע 0.8. אלה בדיקות בתוך תהליך הפיתוח ולא שלוש בחינות סוף עצמאיות.

</div>

In [ ]:
import numpy as np
scores = np.array([0.8, 0.9, 0.7])
print(scores.mean())

<div dir="rtl" lang="he" style="text-align: right;">

**פלט צפוי:** 0.8000000000000002 (עד שגיאת ייצוג מספרי)

## מדוע זה חשוב?
מדדים, אימות והכללה משותפים ליחידה 5 ולניסויים ב־DL. שומרים על אותה שיטת הערכה גם כאשר המודל משתנה.

</div>

<div dir="rtl" lang="he" style="text-align: right;">

## יישום מודרך — חלק 1
מקור הקוד: ערכה 28 הקיימת. בהמשך עוברים לתרגול עצמאי באותו נושא.

</div>

<div dir="rtl" lang="he" style="text-align: right;">

### חישוב ופירוש
קראו את השמות והיחידות, כתבו ציפייה לפני הרצה ובדקו אותה בפלט. הסבירו מה השתנה ומה נשאר קבוע. אין צורך לשנן פקודה בלי להבין את פעולתה.

</div>

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report

<div dir="rtl" lang="he" style="text-align: right;">

### חישוב ופירוש
קראו את השמות והיחידות, כתבו ציפייה לפני הרצה ובדקו אותה בפלט. הסבירו מה השתנה ומה נשאר קבוע. אין צורך לשנן פקודה בלי להבין את פעולתה.

</div>

In [ ]:
# ציונים דידקטיים מומצאים, לא תוצאות Penguins
illustration = pd.DataFrame({
    "K": [1, 7, 101],
    "Train": [0.99, 0.92, 0.60],
    "Validation": [0.71, 0.90, 0.58]
})
display(illustration)

<div dir="rtl" lang="he" style="text-align: right;">

### תשתית הנתונים והחלוקה
קראו תחילה את היעד ואת שמות המאפיינים. החלוקה שומרת התאמה בין שורות X ל־y. הפרמטר random_state מאפשר שחזור ואינו משפר דיוק בפני עצמו.

</div>

In [ ]:
url = "https://raw.githubusercontent.com/samiabo75/Data-Science-ML-Course/main/Datasets/penguins.csv"
df = pd.read_csv(url)
model_df = df.dropna(subset=["species"]).copy()
features = ["flipper_length_mm","body_mass_g"]
X = model_df[features].copy()
y = model_df["species"].copy()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
print("Outer Train:", len(X_train), "Outer Test:", len(X_test))
display(X_train.isna().sum())

<div dir="rtl" lang="he" style="text-align: right;">

### חישוב ופירוש
קראו את השמות והיחידות, כתבו ציפייה לפני הרצה ובדקו אותה בפלט. הסבירו מה השתנה ומה נשאר קבוע. אין צורך לשנן פקודה בלי להבין את פעולתה.

</div>

In [ ]:
# ההגדרות נקבעות לפני צפייה בהערכת Test
candidates = [1, 3, 5, 15, 51]
metric = "f1_macro"
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = []

# קוד תהליך נתון: המטרה להבין מה נכנס ל-CV ומה נשאר בחוץ
for k in candidates:
    pipe = make_pipeline(
        SimpleImputer(strategy="mean"),
        StandardScaler(),
        KNeighborsClassifier(n_neighbors=k, weights="uniform", metric="euclidean")
    )
    fold_results = cross_validate(
        pipe, X_train, y_train,
        cv=cv, scoring=metric, return_train_score=True, n_jobs=1
    )
    # test_score כאן הוא Validation פנימי של הקיפול, לא ה-Test החיצוני
    results.append([
        k,
        fold_results["train_score"].mean(),
        fold_results["test_score"].mean(),
        fold_results["test_score"].std(ddof=0)
    ])

results_df = pd.DataFrame(results, columns=["K", "train_mean", "cv_mean", "cv_std"])
display(results_df)

<div dir="rtl" lang="he" style="text-align: right;">

### התאמה ויישום
זהו מה לומד פרמטרים ובאילו שורות נעשה fit. ודאו שכל הכנה שנלמדת מנתונים משתמשת באימון המתאים. לאחר מכן פירשו את הפלט במילים.

</div>

In [ ]:
# כלל שנקבע מראש: ממוצע CV הגבוה ביותר; בשוויון מדויק K הקטן יותר
# המיון והגישה לשורה הם קטע עזר נתון
ranked = results_df.sort_values(["cv_mean", "K"], ascending=[False, True])
best_k = int(ranked.iloc[0]["K"])
print("Selected K by CV only:", best_k)

# מתאימים צינור חדש לכל קבוצת האימון החיצונית
final_model = make_pipeline(
    SimpleImputer(strategy="mean"),
    StandardScaler(),
    KNeighborsClassifier(n_neighbors=best_k, weights="uniform", metric="euclidean")
)
final_model.fit(X_train, y_train)

<div dir="rtl" lang="he" style="text-align: right;">

### חישוב ופירוש
קראו את השמות והיחידות, כתבו ציפייה לפני הרצה ובדקו אותה בפלט. הסבירו מה השתנה ומה נשאר קבוע. אין צורך לשנן פקודה בלי להבין את פעולתה.

</div>

In [ ]:
# רק לאחר בחירת K והתאמה מחדש: הערכה אחת על Test החיצוני
predictions = final_model.predict(X_test)
test_accuracy = accuracy_score(y_test, predictions)
labels_species = ["Adelie", "Chinstrap", "Gentoo"]
report = classification_report(
    y_test, predictions, labels=labels_species,
    output_dict=True, zero_division=0
)
report_table = pd.DataFrame(report).T
display(report_table.loc[labels_species, ["precision", "recall", "f1-score", "support"]])
test_macro_f1 = report["macro avg"]["f1-score"]
print("CV selection metric:", metric)
print("Selected model Test macro-F1:", test_macro_f1)
print("Selected model Test Accuracy:", test_accuracy)

<div dir="rtl" lang="he" style="text-align: right;">

## עצירה לפני המעבדה
מה דולף כאשר Scaler נלמד לפני CV על כל Train?

**התוצר הנדרש:** תבנית Pipeline נתונה, טבלת CV ובחירת K. מסבירים כל שלב; אין צורך לשנן את התחביר מזיכרון.

</div>